# 00 Final Bronze Summary

Aggregates Bronze reports and creates the Bronze checkpoint when the gate can proceed. Outcomes are STOP for true blockers, PROCEED_WITH_WARNINGS for Silver-resolvable issues, and PROCEED when all checks are clean.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "00_final_summary"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 00_final_summary
Start time: 2026-06-01 18:01:36.692070
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [2]:
report_files = [
    path for path in sorted((ROOT / "eda" / "bronze" / "outputs" / "reports").glob("*/*.json"))
    if path.parent.name != "00_final_summary"
]
rows = []
for path in report_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "report": path.stem,
        "notebook": payload.get("notebook"),
        "p0_status": payload.get("p0_status", "UNKNOWN"),
        "warning_count": int(payload.get("warning_count", 0) or 0),
        "review_count": int(payload.get("review_count", 0) or 0),
        "path": str(path.relative_to(ROOT)),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_TABLES / "bronze_validation_summary.csv", index=False)
display(summary_df)

,report,notebook,p0_status,warning_count,review_count,path
0,file_integrity,01_file_integrity,PASS,0,0,eda\bronze\outputs\reports\01_file_integrity\f...
1,schema_validation,02_schema_validation,PASS,0,0,eda\bronze\outputs\reports\02_schema_validatio...
2,pk_fk_validation,03_pk_fk_checks,PASS,0,0,eda\bronze\outputs\reports\03_pk_fk_checks\pk_...
3,null_analysis,04_null_analysis,PASS,0,7,eda\bronze\outputs\reports\04_null_analysis\nu...
4,range_validation,05_range_validation,PASS,4,0,eda\bronze\outputs\reports\05_range_validation...
5,temporal_validation,06_temporal_checks,PASS,1,0,eda\bronze\outputs\reports\06_temporal_checks\...


In [3]:
fig = px.bar(summary_df.groupby("p0_status", as_index=False).size(), x="p0_status", y="size", color="p0_status", title="Bronze P0 Gate Summary")
fig.write_html(OUTPUT_CHARTS / "bronze_p0_summary.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "bronze_p0_summary.png")
except Exception:
    pass
fig.show()

In [4]:
failed = summary_df[summary_df["p0_status"] != "PASS"]
warning_total = int(summary_df["warning_count"].sum()) if "warning_count" in summary_df else 0
review_total = int(summary_df["review_count"].sum()) if "review_count" in summary_df else 0
status = "STOP" if not failed.empty else ("PROCEED_WITH_WARNINGS" if warning_total or review_total else "PROCEED")
checkpoint = {
    "status": status,
    "timestamp": datetime.now().isoformat(),
    "p0_checks_passed": int((summary_df["p0_status"] == "PASS").sum()),
    "p0_checks_total": int(len(summary_df)),
    "warning_count": warning_total,
    "review_count": review_total,
    "failed_reports": failed.to_dict("records"),
    "next_step": "Proceed to Silver with documented mitigations" if status == "PROCEED_WITH_WARNINGS" else ("Proceed to Silver layer" if status == "PROCEED" else "Resolve Bronze blockers"),
}
write_report("bronze_validation_complete", checkpoint)
write_insight(
    "00_final_summary.md",
    "Bronze Final Summary",
    f"Bronze gate status: {status}.",
    [f"P0 passed: {checkpoint['p0_checks_passed']}/{checkpoint['p0_checks_total']}", f"Warnings: {warning_total}", f"Review items: {review_total}"],
    [f"{row.report}: {row.p0_status}" for row in failed.itertuples()],
    ["STOP only on missing/corrupt files, missing critical schema, PK duplicates, FK orphans, or technical key nulls.", "Non-blocking warnings must be handled or explicitly documented in Silver."],
    [checkpoint["next_step"]],
)
if status in {"PROCEED", "PROCEED_WITH_WARNINGS"}:
    (CHECKPOINTS / "bronze_completed.txt").write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
else:
    checkpoint_path = CHECKPOINTS / "bronze_completed.txt"
    if checkpoint_path.exists():
        checkpoint_path.unlink()
print(checkpoint)

{'status': 'PROCEED_WITH_WARNINGS', 'timestamp': '2026-06-01T18:01:38.162079', 'p0_checks_passed': 6, 'p0_checks_total': 6, 'warning_count': 5, 'review_count': 7, 'failed_reports': [], 'next_step': 'Proceed to Silver with documented mitigations'}
